# Music Recognition System

Adapted from Babar Ayazifar's EE120 course 


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import pandas as pd
import IPython.display as ipd
import hashlib
import csv

from scipy.io import wavfile
from scipy.ndimage import maximum_filter

In [2]:
def spectrogram_function(wavfile,fs=48000,npereg=4096):
    f1,t1,spectrogram = signal.spectrogram(wavfile,fs=48000,nperseg=4096)
    spectrogram = 20*np.log10(spectrogram+1e-12)
    return f1,t1,spectrogram


In [3]:
def peak_finding(spect, neighborhood_size = 2*(25)+1, amp_thresh=40):
    spectcopy = spect.copy()
    max_spect = maximum_filter(spectcopy,size= neighborhood_size)
    mask = (spectcopy == max_spect)

    spectcopy[(mask==False)] = 0
    spectcopy[spectcopy<=amp_thresh] = 0 

    freq_idx, time_idx = np.nonzero(spectcopy)

    return freq_idx, time_idx

In [4]:
def generate_hash(f1, f2, t1, t2, hash_length = 20):
    td = t2 - t1
    h = hashlib.sha1(("%s|%s|%s" % (str(f1), str(f2), str(td))).encode("utf-8"))
    return h.hexdigest()[0:hash_length], t1

def hashing(t1, f1, time_idx, freq_idx, hashes_per_peak=15):
    times, freqs = t1[time_idx], f1[freq_idx]
    sorted_peaks = sorted(zip(freqs, times), key=lambda x: x[1])
    hashes = []
    for m in range(len(sorted_peaks)):
        for n in range(m+1,hashes_per_peak+m+1):
            if n >= len(sorted_peaks):
                break
            generated_hash = generate_hash(sorted_peaks[m][0],sorted_peaks[n][0],sorted_peaks[m][1],sorted_peaks[n][1])
            hashes.append(generated_hash)

    return hashes

In [15]:
def fingerprint(audio, fs, min_distance=25, amp_thresh=40, hashes_per_peak=15):
    neighborhood_size = 2 * min_distance + 1
    
    audio = np.mean(audio, axis=1)
    
    fn,tn,audio_spect = signal.spectrogram(audio,fs=fs,nperseg=4096)
    
    freq_idxn,time_idxn = peak_finding(audio_spect, neighborhood_size, amp_thresh)
    
    fingerprints = hashing(tn, fn, time_idxn, freq_idxn, hashes_per_peak)
    return fingerprints

In [16]:
def detect(audio, fs, db):
    #db = pd.read_csv("database.csv", header=None, names=["Hash", "time", "Song"])
    hashes = fingerprint(audio, fs)
    db_matches = db[db.Hash.isin(map(lambda x: x[0], hashes))]
    if len(db_matches) == 0:
        print("No Matches")
        return

    counts = db_matches.groupby("Song").size()
    counts = counts / counts.sum()
    return counts.idxmax(), counts.max() * 100